In [0]:
# location to save output files
workspace_dir = '/Workspace/Shared/mcp-character-info/data/export_all'

# items to export
catalog = 'mcp'
table_schema = 'game'

tables_to_export = [
  'affiliated_charcters',
  'affiliations',
  'amg_updates',
  'character_updates',
  'characters',
  'infinity_gems',
  'product_codes',
  'product_to_character'
]

# create list of tables for where clause in SQL queries
table_list_sql = ", ".join([f"'{t}'" for t in tables_to_export])

In [0]:
# loop throught tables and save as csv
for table_name in tables_to_export:
  # create full table name and output path
  full_table_name = f"{catalog}.{table_schema}.{table_name}"
  output_file = f"{workspace_dir}/{table_name}.csv"

  print(f"Exporting {table_name}.")

  # save table to Pandas DataFrame because Spark can't write to the Workspace
  df = spark.table(full_table_name).toPandas()

  # write csv file to workspace directory so it can be pushed to GitHub
  df.to_csv(output_file, index=False, mode='w')

  print(f"✅ Finished exporting {table_name}.")

print(f"\n✅ All CSV files copied to {workspace_dir}")

# Create Data Dictionary

In [0]:
# 1. Table List
tables_df = spark.sql(f"""
    SELECT 
       table_name
      ,table_type
      ,comment
    FROM 
      {catalog}.information_schema.tables
    WHERE 
      table_schema = '{table_schema}'
      AND table_name IN ({table_list_sql})
""")

# 2. Column Detail
columns_df = spark.sql(f"""
    SELECT 
       table_name
      ,column_name
      ,ordinal_position
      ,data_type
      ,is_nullable
      ,comment
    FROM 
      {catalog}.information_schema.columns
    WHERE 
      table_schema = '{table_schema}'
      AND table_name IN ({table_list_sql})
    ORDER BY 
       table_name
      ,ordinal_position
""")

# 3. Primary Keys
pk_df = spark.sql(f"""
    SELECT 
       tc.table_name
      ,kcu.column_name
    FROM 
      {catalog}.information_schema.table_constraints tc
      INNER JOIN {catalog}.information_schema.key_column_usage kcu
        ON tc.constraint_name = kcu.constraint_name
        AND tc.table_schema = kcu.table_schema
    WHERE 
      tc.table_schema = '{table_schema}'
      AND tc.constraint_type = 'PRIMARY KEY'
      AND tc.table_name IN ({table_list_sql})
""")

# 4. Foreign Keys
fk_df = spark.sql(f"""
    SELECT
         tc.table_name child_table
        ,kcu.column_name AS child_column
        ,ccu.table_name AS parent_table
        ,ccu.column_name AS parent_column
    FROM 
      {catalog}.information_schema.table_constraints tc
      INNER JOIN {catalog}.information_schema.key_column_usage kcu
        ON tc.constraint_name = kcu.constraint_name
        AND tc.table_schema = kcu.table_schema
      INNER JOIN {catalog}.information_schema.constraint_column_usage ccu
        ON tc.constraint_name = ccu.constraint_name
    WHERE 
      tc.table_schema = '{table_schema}'
      AND tc.constraint_type = 'FOREIGN KEY'
      AND tc.table_name IN ({table_list_sql})
""")

In [0]:
# convert dataframes to lists
tables = tables_df.collect()
columns = columns_df.collect()
pks = pk_df.collect()
fks = fk_df.collect()

In [0]:
import json
from collections import defaultdict

# --- Organize columns by table ---
cols_by_table = defaultdict(list)
for c in columns:
    cols_by_table[c.table_name].append(c)

pk_by_table = defaultdict(list)
for p in pks:
    pk_by_table[p.table_name].append(p.column_name)

# --- Build JSON data dictionary ---
data_dict = {"catalog": catalog, "schema": table_schema, "tables": [], "relationships": []}

for t in tables:
    table_entry = {
        "name": t.table_name,
        "type": t.table_type,
        "comment": t.comment,
        "columns": []
    }
    for c in cols_by_table[t.table_name]:
        table_entry["columns"].append({
            "name": c.column_name,
            "type": c.data_type,
            "nullable": c.is_nullable == "YES",
            "primary_key": c.column_name in pk_by_table[t.table_name],
            "comment": c.comment
        })
    data_dict["tables"].append(table_entry)

for f in fks:
    data_dict["relationships"].append({
        "child_table": f.child_table,
        "child_column": f.child_column,
        "parent_table": f.parent_table,
        "parent_column": f.parent_column
    })

json_output = json.dumps(data_dict, indent=2)
dbutils.fs.put("/Volumes/mcp/game/export_all/data_dictionary.json", json_output, overwrite=True)
print(json_output)

In [0]:
# --- Build DBML output ---
type_map = {
    "int": "integer", "bigint": "bigint", "string": "varchar",
    "double": "double", "float": "float", "boolean": "boolean",
    "date": "date", "timestamp": "timestamp"
}

dbml_lines = []
for t in tables:
    dbml_lines.append(f"Table {t.table_name} {{")
    for c in cols_by_table[t.table_name]:
        dbml_type = type_map.get(c.data_type.lower(), c.data_type.lower())
        flags = []
        if c.column_name in pk_by_table[t.table_name]:
            flags.append("pk")
        if c.is_nullable == "NO":
            flags.append("not null")
        flag_str = f" [{', '.join(flags)}]" if flags else ""
        note = f" // {c.comment}" if c.comment else ""
        dbml_lines.append(f"  {c.column_name} {dbml_type}{flag_str}{note}")
    dbml_lines.append("}\n")

for f in fks:
    dbml_lines.append(f"Ref: {f.child_table}.{f.child_column} > {f.parent_table}.{f.parent_column}")

dbml_output = "\n".join(dbml_lines)
dbutils.fs.put("/Volumes/mcp/game/export_all//data_dictionary.dbml", dbml_output, overwrite=True)
print(dbml_output)